# Task 1, Step 7: Q3 Mixed-Precision AWQ Under Packed-Byte Budgets

Same AWQ-style quantizer as Q2 (same scales searched per bit width, same group-128 quantizer); only bit widths
change. Allowed bits {2, 3, 4, 8}.

**Allocation unit:** an AWQ scale group of one block (qkv, o, gate_up, down) -> 112 units. Projections that share
a folded scale share a bit width, so folding stays algebraically valid.

**Sensitivity measures (both kept; chosen between on development windows only):**
* **C. Isolated-quantization Delta NLL:** quantize only unit u at b bits (with its AWQ scales) and measure the NLL
  increase on the first 32 calibration windows.
* **D. Calibration output error** of the scaled+quantized unit, sum ||dW X||^2 from the Gram matrix
  (HAWQ-style second-order proxy, already computed by the AWQ search).

**Allocation rule:** multiple-choice knapsack: minimise sum_u cost_u(b_u) subject to total packed bytes
(codes + FP16 scales + packed zero-points + FP16 explicit o_proj scales + all FP16 non-eligible parameters, tied
embedding once) <= budget, solved by Lagrangian relaxation and greedy fill. Budgets are the measured totals of
Q2 uniform INT4 and INT3. Selection between C and D: lowest NLL on the 32 development windows.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task1_compression")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task1", need_public=True, need_prefix_banks=False)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import save_json, load_json
from shared.src.plotting import save_figure
from task1_compression.src.runner import setup_task1
from task1_compression.src import quant, awq, allocation
from task1_compression.src.analysis import load_condition, compare_table, summary_table
from task1_compression.src.evaluate import nll_on_sequences
from task1_compression.src.model_utils import layers_of, proj_module

T = setup_task1(ctx)
model = T.model
search = torch.load(ctx.checkpoints / "awq_scales.pt")
O_MODE = search["o_mode"]
PROBE = T.calib[:32]
GROUPS = list(awq.GROUPS)
UNITS = [f"{li}.{g}" for li in range(28) for g in GROUPS]
BITS = allocation.BITS

## Budgets and per-unit packed bytes

In [ ]:
budgets = {b: load_condition(ctx.results, f"q2_awq_int{b}")["storage"]["packed"]["total_bytes"] for b in (4, 3)}
q2st = load_condition(ctx.results, "q2_awq_int4")["storage"]["packed"]
base_bytes = q2st["other_fp16_bytes"] + q2st["awq_explicit_scale_bytes"]   # identical for every bit assignment
unit_bytes = {}
for u in UNITS:
    li, g = u.split(".")
    layer = layers_of(model)[int(li)]
    unit_bytes[u] = {b: sum(quant.matrix_packed_bytes(*proj_module(layer, p).weight.shape, b)["total"] for p in awq.GROUPS[g]) for b in BITS}
print({b: f"{v/2**20:.2f} MiB" for b, v in budgets.items()}, f"base {base_bytes/2**20:.2f} MiB")

## Sensitivity C: isolated-quantization probe

In [ ]:
probe_path = ctx.results / "q3_unit_probe.json"
if probe_path.exists():
    probe = load_json(probe_path)
else:
    T.snap.restore(model)
    t0 = time.perf_counter()
    base_nll = nll_on_sequences(model, PROBE)
    dnll = {}
    for u in UNITS:
        li, g = u.split(".")
        layer = layers_of(model)[int(li)]
        bk = awq.backup_unit(layer, g)
        dnll[u] = {}
        for b in BITS:
            awq.quantize_unit_(layer, g, search["scales"][u][b], b, O_MODE)
            dnll[u][str(b)] = nll_on_sequences(model, PROBE) - base_nll
            awq.restore_unit(layer, g, bk)
        print(u, {k: round(v, 5) for k, v in dnll[u].items()}, flush=True)
    probe = dict(base_nll=base_nll, dnll=dnll, probe_windows=32, time_s=time.perf_counter() - t0)
    save_json(probe_path, probe)

sens_C = {u: {b: probe["dnll"][u][str(b)] for b in BITS} for u in UNITS}
sens_D = {u: {b: search["log"][u][b]["err"] for b in BITS} for u in UNITS}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (nm, S) in zip(axes, (("C: isolated dNLL @2-bit", sens_C), ("D: log10 calib. output error @2-bit", sens_D))):
    M = pd.DataFrame({g: [S[f"{li}.{g}"][2] for li in range(28)] for g in GROUPS})
    sns.heatmap(M if nm.startswith("C") else np.log10(M), ax=ax, cmap="rocket_r"); ax.set_title(nm); ax.set_ylabel("block")
save_figure(ctx.figures, "q3_sensitivity")

## Candidate allocations, budget check, development-set selection

In [ ]:
cand_path = ctx.results / "q3_candidates.json"
cands = load_json(cand_path) if cand_path.exists() else {}
n_layers = 28
for bud_bits, budget in budgets.items():
    for sname, sens in (("C_dnll", sens_C), ("D_proxy", sens_D)):
        tag = f"int{bud_bits}_{sname}"
        if tag in cands:
            continue
        units = {u: dict(cost=sens[u], bytes=unit_bytes[u]) for u in UNITS}
        r = allocation.allocate_bits(units, budget, base_bytes=base_bytes)
        ub = r["assignment"]
        T.snap.restore(model)
        t0 = time.perf_counter()
        st = awq.apply_awq_(model, search["scales"], ub, o_mode=O_MODE)
        total = quant.model_packed_bytes(model, awq.projection_bits(ub, n_layers), st["explicit_scale_channels"])["total_bytes"]
        assert total <= budget, (total, budget)
        cands[tag] = dict(budget_bits=bud_bits, sensitivity=sname, unit_bits=ub, total_bytes=total, budget=budget,
                          dev_nll=nll_on_sequences(model, T.dev), time_s=time.perf_counter() - t0)
        save_json(cand_path, cands)
        print(tag, f"dev NLL {cands[tag]['dev_nll']:.4f}", f"bytes {total/2**20:.2f}/{budget/2**20:.2f} MiB", flush=True)
# uniform Q2 reference on dev
for b in (4, 3):
    tag = f"int{b}_uniform_q2"
    if tag not in cands:
        T.snap.restore(model)
        awq.apply_awq_(model, search["scales"], awq.unit_bits_uniform(28, b), o_mode=O_MODE)
        cands[tag] = dict(budget_bits=b, sensitivity="uniform", dev_nll=nll_on_sequences(model, T.dev), total_bytes=budgets[b], budget=budgets[b])
        save_json(cand_path, cands)
cdf = pd.DataFrame(cands).T
cdf[["budget_bits", "sensitivity", "dev_nll", "total_bytes", "budget"]]

In [ ]:
selected = {}
for b in (4, 3):
    sub = {k: v for k, v in cands.items() if v["budget_bits"] == b and v["sensitivity"] != "uniform"}
    selected[b] = min(sub, key=lambda k: sub[k]["dev_nll"])
    print(f"INT{b} budget: selected {selected[b]}")
save_json(ctx.results / "q3_selection.json", dict(rule="lowest NLL on 32 C4-train development windows", selected=selected))

## Final Q3 runs

In [ ]:
for b in (4, 3):
    ub = cands[selected[b]]["unit_bits"]
    T.runner.run(f"q3_budget_int{b}",
                 apply_fn=lambda m, ub=ub: awq.apply_awq_(m, search["scales"], ub, o_mode=O_MODE),
                 storage_fn=lambda m, st, ub=ub: dict(packed=quant.model_packed_bytes(m, awq.projection_bits(ub, 28), st["explicit_scale_channels"])),
                 meta=dict(method="AWQ-style mixed precision", unit="AWQ scale group", candidate=selected[b], unit_bits=ub,
                           budget_bytes=budgets[b]))

In [ ]:
for b in (4, 3):
    q3 = load_condition(ctx.results, f"q3_budget_int{b}")["storage"]["packed"]
    print(f"INT{b}: Q3 {q3['total_bytes']:,} B <= Q2 {budgets[b]:,} B : {q3['total_bytes'] <= budgets[b]};"
          f" bits/eligible weight {q3['bits_per_eligible_weight']:.3f} (nominal {q3['nominal_bits']:.3f})")
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
for ax, b in zip(axes, (4, 3)):
    ub = cands[selected[b]]["unit_bits"]
    M = pd.DataFrame({g: [ub[f"{li}.{g}"] for li in range(28)] for g in GROUPS})
    sns.heatmap(M, ax=ax, annot=True, cbar=False, cmap="viridis", vmin=2, vmax=8); ax.set_title(f"Q3 bits, INT{b} budget"); ax.set_ylabel("block")
save_figure(ctx.figures, "q3_bit_maps")
pd.DataFrame({b: pd.Series(cands[selected[b]]["unit_bits"]).value_counts() for b in (4, 3)})

In [ ]:
compare_table(ctx.results, [("q3_budget_int4", "q2_awq_int4"), ("q3_budget_int3", "q2_awq_int3"),
                            ("q3_budget_int4", "fp16"), ("q3_budget_int3", "fp16")])

In [ ]:
summary_table(ctx.results).loc[["fp16", "q2_awq_int4", "q3_budget_int4", "q2_awq_int3", "q3_budget_int3"]]